In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Sonia_Vihar_Delhi_DPCC_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,350.0,217.0,214.0,106.0,176.0,249.0,231.0,49.0,83.0,153.0,378.0,296.0
1,2,335.0,240.0,152.0,120.0,162.0,172.0,118.0,92.0,93.0,197.0,366.0,301.0
2,3,321.0,231.0,125.0,171.0,222.0,140.0,124.0,83.0,115.0,153.0,NaN,290.0
3,4,382.0,301.0,168.0,167.0,247.0,188.0,71.0,87.0,64.0,160.0,398.0,157.0
4,5,339.0,236.0,116.0,143.0,301.0,268.0,101.0,62.0,69.0,129.0,386.0,138.0
5,6,182.0,140.0,112.0,151.0,233.0,182.0,49.0,78.0,123.0,NaN,376.0,217.0
6,7,302.0,179.0,168.0,165.0,318.0,201.0,50.0,70.0,88.0,NaN,414.0,250.0
7,8,296.0,151.0,129.0,186.0,233.0,222.0,57.0,73.0,94.0,196.0,401.0,327.0
8,9,272.0,137.0,108.0,NaN,172.0,219.0,97.0,70.0,NaN,167.0,374.0,228.0
9,10,288.0,267.0,150.0,224.0,172.0,212.0,265.0,80.0,91.0,124.0,NaN,265.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,350.000000,217.000000,214.0,106.00000,176.000000,159.342857,109.583333,72.6875,83.000000,153.000000,378.000000,296.000000
1,2,335.000000,240.000000,152.0,120.00000,162.000000,172.000000,118.000000,92.0000,93.000000,197.000000,366.000000,301.000000
2,3,321.000000,231.000000,125.0,171.00000,222.000000,140.000000,124.000000,83.0000,115.000000,153.000000,348.129032,290.000000
3,4,382.000000,301.000000,168.0,167.00000,247.000000,188.000000,71.000000,87.0000,64.000000,160.000000,398.000000,157.000000
4,5,339.000000,236.000000,116.0,143.00000,301.000000,159.342857,101.000000,62.0000,69.000000,129.000000,386.000000,138.000000
5,6,182.000000,140.000000,112.0,151.00000,233.000000,182.000000,49.000000,78.0000,123.000000,217.529412,376.000000,217.000000
6,7,302.000000,179.000000,168.0,165.00000,318.000000,201.000000,50.000000,70.0000,88.000000,217.529412,414.000000,250.000000
7,8,296.000000,151.000000,129.0,186.00000,233.000000,222.000000,57.000000,73.0000,94.000000,196.000000,401.000000,327.000000
8,9,272.000000,137.000000,108.0,158.21875,172.000000,219.000000,97.000000,70.0000,93.333333,167.000000,374.000000,228.000000
9,10,288.000000,267.000000,150.0,224.00000,172.000000,212.000000,109.583333,80.0000,91.000000,124.000000,348.129032,265.000000
